# Ticket 13: local_amount source-data investigation

Ticket 6 found 20 documents in P01 where `local_amount` broadcasts the
document total across every debit line. Mission 10 confirmed the same
shape in P02/P03. This notebook checks two things ticket 6 never had
reason to check: is line count really the cause, and is company 1000 /
FY2024 really the scope - by querying `stg_gl` with no scope filter at
all, since it holds the full source file regardless of this pipeline's
declared scope.

In [1]:
import duckdb  # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
con.execute("SELECT COUNT(*), COUNT(DISTINCT company_code), MIN(fiscal_year), MAX(fiscal_year) FROM stg_gl").fetchall()

[(648801, 4, 2024, 2025)]

## Is high line count really the cause?

Ticket 6's hypothesis: documents with unusually high line counts (18+)
get the broadcast defect. Check every `source` value that produces
high-line-count documents, not just the ones already flagged.

In [2]:
q = """
WITH doc AS (
    SELECT company_code, document_id, fiscal_year, fiscal_period,
           ANY_VALUE(source) AS source, COUNT(*) AS n_lines,
           ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
           ROUND(SUM(local_amount), 2) AS net_local
    FROM stg_gl GROUP BY 1, 2, 3, 4
)
SELECT source, COUNT(*) AS n_docs, ROUND(AVG(n_lines), 1) AS avg_lines,
       MAX(n_lines) AS max_lines,
       COUNT(*) FILTER (WHERE n_lines >= 18) AS n_high_line_docs,
       COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01) AS n_defect
FROM doc GROUP BY 1 ORDER BY n_docs DESC
"""
con.execute(q).fetchdf()

,source,n_docs,avg_lines,max_lines,n_high_line_docs,n_defect
0,RV,24179,3.5,202,248,2
1,KR,18265,3.4,192,154,1
2,DR,15162,3.4,212,155,2
3,SA,13839,3.4,200,112,0
4,DZ,12107,3.5,202,115,0
...,...,...,...,...,...,...
521,Z412,2,2.0,2,0,0
522,Z465,1,2.0,2,0,0
523,Z425,1,2.0,2,0,0
524,Z487,1,6.0,6,0,0


Every common `source` value produces plenty of 18+-line documents
(hundreds each), and almost none of them are defective. High line count
is common across the whole file; the defect isn't. Line count alone
doesn't explain it.

## What does explain it: source = 'AB'

In [3]:
q = """
WITH doc AS (
    SELECT company_code, document_id, fiscal_year, fiscal_period,
           ANY_VALUE(source) AS source, COUNT(*) AS n_lines,
           ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
           ROUND(SUM(local_amount), 2) AS net_local
    FROM stg_gl GROUP BY 1, 2, 3, 4
)
SELECT
  COUNT(*) AS n_ab_docs,
  MIN(n_lines) AS min_lines, MAX(n_lines) AS max_lines, ROUND(AVG(n_lines), 1) AS avg_lines,
  COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01) AS n_defect
FROM doc WHERE source = 'AB'
"""
con.execute(q).fetchdf()

,n_ab_docs,min_lines,max_lines,avg_lines,n_defect
0,1123,31,155,57.0,1123


Every single `source = 'AB'` document is defective. No exceptions.
`AB` documents also never have fewer than 31 lines - they're
structurally high-line-count, which is why ticket 6's line-count
hypothesis looked right from inside the P01 scope. Line count is
downstream of the real cause, not the cause.

## Does source = 'AB' fully explain P01/P02/P03's already-known counts?

In [4]:
for p in (1, 2, 3):
    q = f"""
    WITH doc AS (
        SELECT document_id, ANY_VALUE(source) AS source, COUNT(*) AS n_lines,
               ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
               ROUND(SUM(local_amount), 2) AS net_local
        FROM stg_gl WHERE company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = {p}
        GROUP BY 1
    )
    SELECT COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01) AS n_defect,
           COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01 AND source = 'AB') AS n_defect_and_ab
    FROM doc
    """
    print(f"P{p:02d}", con.execute(q).fetchone())

P01 (20, 19)
P02 (27, 27)
P03 (36, 36)


P02 and P03 match exactly (27/27, 36/36). P01 is 19/20 - the one
exception is `source = 'RV'`, net `local_amount` of `0.02`, already
known from mission 06 as one of the 3 genuine sub-cent rounding cases,
not this defect. `source = 'AB'` explains every real broadcast-defect
document across all three periods already checked.

## Full scope: every company, every fiscal year

In [5]:
q = """
WITH doc AS (
    SELECT company_code, document_id, fiscal_year, fiscal_period,
           ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
           ROUND(SUM(local_amount), 2) AS net_local
    FROM stg_gl GROUP BY 1, 2, 3, 4
)
SELECT company_code, fiscal_year,
       COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01) AS n_docs,
       ROUND(SUM(net_local) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01), 2) AS total
FROM doc GROUP BY 1, 2 ORDER BY 1, 2
"""
con.execute(q).fetchdf()

,company_code,fiscal_year,n_docs,total
0,1000,2024,296,3.980025e+09
1,1000,2025,7,1.277905e+08
2,2000,2024,272,1.625495e+09
3,2000,2025,4,2.364434e+06
4,2100,2024,278,1.957924e+09
5,2100,2025,5,1.011853e+08
6,3000,2024,264,1.706874e+09
7,3000,2025,6,2.109059e+07


In [6]:
grand = con.execute("""
WITH doc AS (
    SELECT company_code, document_id, fiscal_year, fiscal_period,
           ROUND(SUM(debit_amount) - SUM(credit_amount), 2) AS dc_gap,
           ROUND(SUM(local_amount), 2) AS net_local
    FROM stg_gl GROUP BY 1, 2, 3, 4
)
SELECT COUNT(*) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01),
       ROUND(SUM(net_local) FILTER (WHERE dc_gap = 0 AND ABS(net_local) > 0.01), 2)
FROM doc
""").fetchone()
scale = con.execute("SELECT ROUND(SUM(debit_amount), 2), ROUND(SUM(ABS(local_amount)), 2) FROM stg_gl").fetchone()
grand, scale

((1132, 9522749609.33), (45257592963.24, 88542897320.76))

All 4 companies, both fiscal years. 1,132 documents, `$9,522,749,609.33`
net `local_amount`. Total `debit_amount` across the whole file is
`$45,257,592,963.24`; total `|local_amount|` is `$88,542,897,320.76` -
this defect touches roughly a fifth of total debit activity by that
measure. Not a P01 finding, not a company-1000 finding: dataset-wide,
same signature, `source = 'AB'` every time.

Root cause, confirmed scope, and a decision to record are written up in
`docs/missions/13-local-amount-source-investigation.md`. Nothing in
`stg_gl` or any loaded table changes as a result of this notebook.